In [1]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv()

DATA_DIR            = Path("../data")
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "mlruns")
EXPERIMENT_NAME     = "csgo-hp-tuning"
N_TRIALS            = 50
N_SPLITS            = 5

In [2]:
import pandas as pd

X_train = pd.read_csv(DATA_DIR / "X_train.csv", index_col=0)
y_train = pd.read_csv(DATA_DIR / "y_train.csv", index_col=0).squeeze()

print(f"Loaded {len(X_train)} rows, {X_train.shape[1]} features")

Loaded 36618 rows, 16 features


In [3]:
import mlflow
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

tscv = TimeSeriesSplit(n_splits=N_SPLITS)

def objective(trial: optuna.Trial) -> float:
    C        = trial.suggest_float("C", 1e-3, 100, log=True)
    solver   = trial.suggest_categorical("solver", ["lbfgs", "liblinear"])
    max_iter = trial.suggest_int("max_iter", 200, 2000, step=200)

    model  = LogisticRegression(C=C, solver=solver, max_iter=max_iter)
    scores = cross_val_score(model, X_train, y_train, cv=tscv, scoring="roc_auc", n_jobs=-1)
    auc    = float(scores.mean())

    with mlflow.start_run(nested=True):
        mlflow.log_params({"C": C, "solver": solver, "max_iter": max_iter})
        mlflow.log_metric("cv_roc_auc", auc)

    return auc

/home/dtetu/Documents/MLOps/08_hyperparameter_tunning/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/dtetu/Documents/MLOps/08_hyperparameter_tunning/.venv/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/19 10:12:40 INFO mlflow.tracking.fluent: Experiment with name 'csgo-hp-tuning' does not exist. Creating a new experiment.


In [4]:
with mlflow.start_run(run_name="optuna-search"):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"Best ROC AUC : {study.best_value:.4f}")
print(f"Best params  : {study.best_params}")

[I 2026-05-19 10:12:40,589] A new study created in memory with name: no-name-0231963b-c3ff-40b3-8c08-cd4e5f580e39


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:02<?, ?it/s]

Best trial: 0. Best value: 0.809481:   0%|          | 0/50 [00:02<?, ?it/s]

Best trial: 0. Best value: 0.809481:   2%|▏         | 1/50 [00:02<01:38,  2.01s/it]

...

Best trial: 5. Best value: 0.809498:  98%|█████████▊| 49/50 [00:14<00:00,  4.93it/s]

Best trial: 5. Best value: 0.809498: 100%|██████████| 50/50 [00:14<00:00,  4.99it/s]

Best trial: 5. Best value: 0.809498: 100%|██████████| 50/50 [00:14<00:00,  3.49it/s]

[I 2026-05-19 10:12:54,723] Trial 48 finished with value: 0.8094761788591025 and parameters: {'C': 0.0266360753158956, 'solver': 'liblinear', 'max_iter': 1000}. Best is trial 5 with value: 0.8094975212254842.
[I 2026-05-19 10:12:54,920] Trial 49 finished with value: 0.8094500640826723 and parameters: {'C': 0.007398363610303003, 'solver': 'lbfgs', 'max_iter': 1600}. Best is trial 5 with value: 0.8094975212254842.


Best ROC AUC : 0.8095
Best params  : {'C': 0.012775784421067391, 'solver': 'liblinear', 'max_iter': 1600}


In [5]:
import json

output   = {"params": study.best_params, "cv_roc_auc": study.best_value}
out_path = DATA_DIR / "best_params.json"

with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved to {out_path}")
print("Re-run step 03 to train a final model with these hyperparameters.")

Saved to ../data/best_params.json
Re-run step 03 to train a final model with these hyperparameters.
